# Full `DPR processing` Prefect Flow

  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-797
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-798
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-799
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-800
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-821
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-852
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-854
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-869
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-871   

These stories implement a new "DPR processing" Prefect flow that, for a given processor and arguments, will:

  1. Stage the needed cadip session.
  1. Retrieve the tasktable.
  1. Calculate the processing unit list.
  1. For each processing unit:
      1. Calculate the CQL2 filters to find the auxiliary files.
      1. Stage the auxiliary files.
  1. Calculate the payload file
  1. Run the processor
  1. Publish its results into the catalog

## Initialisation

In [ ]:
# Imports
import ast
from IPython.display import JSON
import os
import os.path as osp

from resources.widget_utils import *

from rs_client.ogcapi.dpr_client import DprProcessor
from rs_common.prefect_utils import *
from rs_workflows.auxip_flow import auxip_staging
from rs_workflows.flow_utils import  DprProcessIn, Priority, ProcessingMode, WorkflowType
from rs_workflows.init_pi_db_flow import init_pi_database
from rs_workflows.on_demand_processing import dpr_processing, on_demand_cadip_staging

In [ ]:
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard_url = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard_url}")

In [ ]:
# Choose prefect deployment method
deploy_prefect_radio

In [ ]:
# Choose prefect flow run method
run_prefect_radio

In [ ]:
# Choose dpr processor
dpr_proc_radio

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_staging(scale=2)

# Init the processor dask cluster. 
from resources import dask_utils
dask_utils.cluster_info_eopf = ClusterInfo("jupyter_token", "cluster_label", "cluster_instance")
print(f"** Init Dask cluster for: {dpr_proc_radio.value!r} **")
match dpr_proc_radio.value:
    case "mockup":
        init_dask_cluster_mockup(scale=1)
    case DprProcessor.S1L0.value | DprProcessor.S3L0.value:
        init_dask_cluster_l0(scale=1,
                                big_resources=True,  # provide more ram and cpu
                                worker_cores=4,      # number of CPU per worker 
                                worker_memory=58,    # memory per worker in GB
        )
    case DprProcessor.S1ARD.value:
        init_dask_cluster_s1ard(scale=1)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_staging)
display(dask_cluster_eopf)

In [ ]:
# Get the prefect share bucket folder
share_bucket, _ = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")
s3_payload_file = osp.join(s3_config, f"payload_file_{dpr_proc_radio.value}_{OWNER_ID}.yaml")
s3_code_folder = f"users/{OWNER_ID}/code"

In [ ]:
# Create test collections
INPUT_COLLECTION = "TEST_FLOW_INPUT"
AUXIP_COLLECTION = "TEST_FLOW_AUXIP"
OUTPUT_COLLECTION = "TEST_FLOW_OUTPUT"
if dpr_proc_radio.value in (DprProcessor.S1L0.value, DprProcessor.S3L0.value):
    if dpr_proc_radio.value == DprProcessor.S1L0.value:
        OUTPUT_COLLECTION = "TEST_FLOW_OUTPUT_S1"
    else:
        OUTPUT_COLLECTION = "TEST_FLOW_OUTPUT_S3"
        
for collection in (INPUT_COLLECTION, AUXIP_COLLECTION, OUTPUT_COLLECTION):
  create_test_collection(collection)
  
# Prefect flow environment arguments
flow_env_args = {
  "env": {
    "owner_id": OWNER_ID,
  },
}

# DPR processing input parameters
dpr_process_in = DprProcessIn(
    **flow_env_args,         
    processor_name=dpr_proc_radio.value, 
    processor_version="", # NOTE: is it used ?
    dask_cluster_label=cluster_info_eopf.cluster_label,
    # TODO: the following param was added extra from https://pforge-exchange2.astrium.eads.net/confluence/pages/viewpage.action?pageId=477103370
    # We couldn't get the exact info on how to build the path were the payload should be written
    s3_payload_file = s3_payload_file,    
    # end of TODO
    pipeline = "set_me_later",
    unit = "",
    priority = Priority.LOW,
    workflow_type = WorkflowType.ON_DEMAND,
    input_products=[{"set_me_later": ("set_me_later", INPUT_COLLECTION)}],
    auxiliary_product_to_collection_identifier = {"*": AUXIP_COLLECTION},
    generated_product_to_collection_identifier = [{"set_me_later": ("set_me_later", OUTPUT_COLLECTION)}],
    processing_mode = [ProcessingMode.ALWAYS],
    start_datetime="2014-01-01T11:00:00Z",
    end_datetime="2025-10-03T11:00:00Z",
    satellite=None,
)

## Deploy and run INIT PI DB flow

In [ ]:
# Deploy the Prefect flow
pi_deploy = await deploy_prefect(
    deploy_file="../../sprint27/init_pi_db_flows.yaml", 
    s3_code_folder=s3_code_folder, 
    work_pool_name=os.environ["PREFECT_WORK_POOL_GENERAL"]
)

In [ ]:
# Run the Prefect flow
await run_prefect(
    deploy_name=pi_deploy, 
    py_func=init_pi_database, 
    params=flow_env_args
)

## Deploy rs-client-libraries Prefect flows

In [ ]:
# Deploy the Prefect flows
dpr_processing_deploy, auxip_staging_deploy, cadip_staging_deploy = await deploy_prefect(
    deploy_file="./dpr_processing_flow.yaml", 
    s3_code_folder=s3_code_folder, 
    work_pool_name=os.environ["PREFECT_WORK_POOL_EOPF"]
)

## Init the L0 demos

In [ ]:
if dpr_proc_radio.value in (DprProcessor.S1L0.value, DprProcessor.S3L0.value):
    print(f"Init demo for: {dpr_proc_radio.value!r}")

    if dpr_proc_radio.value == DprProcessor.S1L0.value:        
        dpr_process_in.satellite = "sentinel-1a"
        cadip_collection = "cadip_sentinel1"
        # Use the following session for the short set of cadip data
        cadip_session = "S1A_20250611050700059595"
        # Use the following session for the full set of cadip data
        #cadip_session = "S1A_20250611050700059594"
        # Update the input product list of the dpr processing
        dpr_process_in.input_products = [{"S1CADUS": (cadip_session, INPUT_COLLECTION)}]
        dpr_process_in.generated_product_to_collection_identifier = [{"output_folder": ("S01SIWRAW", OUTPUT_COLLECTION)}]
    else:        
        dpr_process_in.satellite = "sentinel-3a"
        cadip_collection = "cadip_sentinel3"
        # Use the following session for the short set of cadip data
        cadip_session = "S3A_20200121061417020456"
        # Use the following session for the full set of cadip data
        #cadip_session = "S3A_20200121061417020455"        
        # Update the input product list of the dpr processing
        dpr_process_in.input_products = [{"S3ACADUS": (cadip_session, INPUT_COLLECTION)}]
        dpr_process_in.generated_product_to_collection_identifier = [{"S03ISPS": ("*", OUTPUT_COLLECTION)},
                                                                     {"S03CACHE": ("S03OLCL0_", OUTPUT_COLLECTION)}
                                                                     ]

    # Stage a cadip session
    params = {
        **flow_env_args,
        "cadip_collection_identifier": cadip_collection,
        "session_identifier": cadip_session,
        "catalog_collection_identifier": INPUT_COLLECTION,
    }    
    await run_prefect(cadip_staging_deploy, on_demand_cadip_staging, params)    

## Init the S1-ARD demo

<div class="alert alert-block alert-warning">

**NOTE**: for the S1-ARD demo initialization, we need to stage products from the PRIP station. 

The corresponding Prefect flow is not implemented yet so we do it directly from the python client.

**SEE ALSO** Pierre's issue on the S1-ARD API: https://gitlab.eopf.copernicus.eu/S1/s1-ard-core/-/issues/21
</div>

In [ ]:
if dpr_proc_radio.value == DprProcessor.S1ARD.value:
    print(f"Init demo for: {dpr_proc_radio.value!r}")
    dpr_process_in.satellite = "sentinel-1a"
    dpr_process_in.input_products = []
    dpr_process_in.generated_product_to_collection_identifier = []

    for idx, id in enumerate([
        "S1A_IW_SLC__1SDV_20240428T171518_20240428T171545_053637_068367_71A2.SAFE.short.zip",
        "S1A_IW_SLC__1SDV_20240416T171518_20240416T171545_053462_067C88_DA9D.SAFE.short.zip"
    ]):
        # Search item
        found = prip_client.search(method='GET', stac_filter=f"Name={id!r}").to_dict()
        display(found)

        print(f"Stage Prip product:\n{json.dumps(found, indent=2)}")
        stage_data(found, [id], INPUT_COLLECTION)

        # Update the input product list of the dpr processing
        dpr_process_in.input_products.append({f"S1PRIP{idx}": (id, INPUT_COLLECTION)})
        # TODO: find the product type in S1ARD processor case and use it in the following
        dpr_process_in.generated_product_to_collection_identifier.append[{"output_folder": ("product_type", OUTPUT_COLLECTION)}]

## Read the tasktable

<div class="alert alert-block alert-warning">

Note: for now the tasktables are hardcoded, not returned by the processors.
</div>

In [ ]:
tasktable: dict = dpr_client.get_process(dpr_proc_radio.value, cluster_info_eopf)
print(f"Tasktable for {dpr_proc_radio.value!r}:")
display(JSON(tasktable))
# print(json.dumps(tasktable, indent=2))

## Choose calling parameters

In [ ]:
pipeline_unit_radio = get_pipeline_unit_radio()
pipeline_unit_radio

## Run the DPR processing flow

<div class="alert alert-block alert-warning">

Notes: 

  * **S1L0** and **S3L0** fail because of https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-812
  * **S1-ARD** is failing because we need to discuss how to handle its tasktable ('io/alternatives' is missing)
</div>

In [ ]:
# Update the input parameters from this notebook radio buttons
dpr_process_in.processor_name=dpr_proc_radio.value
dpr_process_in.dask_cluster_label=cluster_info_eopf.cluster_label
dpr_process_in.__dict__.update(pipeline_unit_radio.value)

print(f"Run demo for: {dpr_proc_radio.value!r}")

# Run the processor
params = {"dpr_input": dpr_process_in.model_dump(mode="json")}
state = await run_prefect(
    deploy_name=dpr_processing_deploy, 
    py_func=dpr_processing, 
    params=params
)
if state is not None:
    flow_run_id = state.state_details.flow_run_id
    print(f"Flow run id: {flow_run_id!r}")
    if state.is_failed() or state.is_crashed():
        # Retrieve logs of failed flow run
        response = http_session.get(f"{os.environ['PREFECT_API_URL']}/flow_runs/{flow_run_id}/logs/download")
        response.raise_for_status()
        logs_text = response.text

        print("=== Prefect flow logs ===")
        print(logs_text)
        print("=== End of logs ===")

        raise RuntimeError(f"Prefect flow {flow_run_id} failed.\n\nLogs:\n{logs_text}")

## Read artifacts

In [ ]:
# Get the processing unit list from the last flow run artifacts
# See: https://docs-3.prefect.io/v3/api-ref/rest-api/server/artifacts/read-latest-artifact
response = http_session.get(f"{os.environ['PREFECT_API_URL']}/artifacts/processing-unit-list/latest")
response.raise_for_status()
contents = response.json()["data"]

# Render the artifact as markdown
from IPython.display import Markdown
display(Markdown(contents))

In [ ]:
# Also get the latest auxip cqlS filter that was used for auxip staging
response = http_session.get(f"{os.environ['PREFECT_API_URL']}/artifacts/auxip-cql2/latest")
response.raise_for_status()
contents = response.json()["data"]

# Render the artifact as markdown
from IPython.display import Markdown
display(Markdown(contents))

# Extract the cql2 dict between the ``` from the markdown
start = "```json"
end = "```"
json_contents = contents[contents.find(start)+len(start):contents.rfind(end)]
cql2_filter = ast.literal_eval(json_contents)

In [ ]:
# Get the payload file from the last flow run artifacts
# See: https://docs-3.prefect.io/v3/api-ref/rest-api/server/artifacts/read-latest-artifact
response = http_session.get(f"{os.environ['PREFECT_API_URL']}/artifacts/dpr-payload-file/latest")
response.raise_for_status()
contents = response.json()["data"]

# Render the artifact as markdown
from IPython.display import Markdown
display(Markdown(contents))

## Manual call to Auxip staging

In [ ]:
# We can call manually the auxip staging with this cql2 filter
params = {
    **flow_env_args,
    "cql2_filter": cql2_filter,
    "catalog_collection_identifier": AUXIP_COLLECTION,
}
await run_prefect(
    deploy_name=auxip_staging_deploy, 
    py_func=auxip_staging, 
    params=params
)

# Then get the staged items from the last flow run artifacts
response = http_session.get(f"{os.environ['PREFECT_API_URL']}/artifacts/auxiliary-files/latest")
response.raise_for_status()
contents = response.json()["data"]

# Render the artifact as markdown
from IPython.display import Markdown
display(Markdown(f"```json\n{contents}\n```"))

## Shutdown the dask clusters

In [ ]:
# Choose to shutdown the dask cluster
shutdown_checkbox

In [ ]:
if shutdown_checkbox.value:
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)
    close_dask_clusters()
# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.